# Minisimulación de recuperación y similitud coseno

Este notebook reproduce una versión pequeña del workflow de Seminario 1:

```text
Query → preprocesamiento → representación TF-IDF → vectores
→ similitud coseno → ranking de evidencias
```

El corpus utilizado es sintético y sirve únicamente para demostrar el procedimiento. No se está determinando si una afirmación médica es verdadera o falsa. La similitud coseno expresa relación entre textos, no veracidad.

## Alcance de la demostración

- Incluye una query de prueba y cuatro documentos de evidencia de demostración.
- Implementa el preprocesamiento y TF-IDF manualmente con NumPy, sin `scikit-learn`.
- Calcula la similitud coseno entre la query y cada documento.
- Devuelve un ranking Top-k ordenado por similitud.
- Incluye una celda opcional para probar SBERT real si se instala `sentence-transformers`.
- No incluye API externa, RAG ni clasificación de veracidad.
- Incluye una simulación didáctica de Triplet Loss y un bloque opcional para SBERT real.

Si NumPy no está instalado en el entorno del notebook, ejecutar previamente `%pip install numpy`.

In [1]:
from collections import Counter
import math
import re

import numpy as np

np.set_printoptions(precision=4, suppress=True)
print(f"NumPy {np.__version__} cargado correctamente")

NumPy 2.5.3 cargado correctamente


## 1. Query y corpus sintético

Los textos siguientes no representan fuentes médicas reales. Se incluyen solo para observar cómo cambia el puntaje cuando los términos de la query aparecen en documentos relacionados o irrelevantes.

In [2]:
query = "¿Las vacunas causan malestar estomacal?"

documents = [
    {
        "id": "E1",
        "title": "Evidencia demo sobre vacunas y molestias digestivas",
        "text": "Documento de demostración sobre si las vacunas causan malestar estomacal y otros efectos digestivos.",
    },
    {
        "id": "E2",
        "title": "Evidencia demo sobre efectos secundarios de vacunación",
        "text": "Documento de demostración sobre vacunas y efectos secundarios generales después de recibir una vacuna.",
    },
    {
        "id": "E3",
        "title": "Evidencia demo sobre alimentación y malestar",
        "text": "Documento de demostración sobre alimentos y malestar estomacal, sin relación con vacunas.",
    },
    {
        "id": "E4",
        "title": "Evidencia demo sobre deporte",
        "text": "Documento de demostración sobre resultados de un partido de fútbol y entrenamiento deportivo.",
    },
]

print("Query de prueba:", query)
print(f"Documentos de demostración: {len(documents)}")

Query de prueba: ¿Las vacunas causan malestar estomacal?
Documentos de demostración: 4


## 2. Preprocesamiento textual

Se convierten los textos a minúsculas, se eliminan signos de puntuación, se tokenizan y se retiran algunas palabras funcionales frecuentes. No se aplica lematización para mantener la demostración transparente.

In [3]:
STOPWORDS_ES = {
    "a", "al", "con", "como", "de", "del", "el", "en",
    "es", "esta", "este", "la", "las", "los", "más",
    "para", "por", "que", "se", "sin", "sobre", "un", "una",
}

def normalize_and_tokenize(text):
    normalized = text.lower()
    normalized = re.sub(r"[^a-záéíóúüñ0-9\s]", " ", normalized)
    tokens = re.findall(r"[a-záéíóúüñ0-9]+", normalized)
    return [token for token in tokens if token not in STOPWORDS_ES and len(token) > 1]

query_tokens = normalize_and_tokenize(query)
document_tokens = [normalize_and_tokenize(document["text"]) for document in documents]

print("Tokens de la query:", query_tokens)
for document, tokens in zip(documents, document_tokens):
    print(f"{document['id']}: {tokens}")

Tokens de la query: ['vacunas', 'causan', 'malestar', 'estomacal']
E1: ['documento', 'demostración', 'si', 'vacunas', 'causan', 'malestar', 'estomacal', 'otros', 'efectos', 'digestivos']
E2: ['documento', 'demostración', 'vacunas', 'efectos', 'secundarios', 'generales', 'después', 'recibir', 'vacuna']
E3: ['documento', 'demostración', 'alimentos', 'malestar', 'estomacal', 'relación', 'vacunas']
E4: ['documento', 'demostración', 'resultados', 'partido', 'fútbol', 'entrenamiento', 'deportivo']


## 3. Construcción manual de TF-IDF

Para cada término se calcula una frecuencia inversa de documento suavizada:

$$
idf(t) = \log\left(\frac{1 + N}{1 + df(t)}\right) + 1
$$

La query y los documentos se representan con el mismo vocabulario para que sus vectores puedan compararse.

In [4]:
# El IDF se calcula únicamente sobre los documentos candidatos, no sobre la query.
vocabulary = sorted({token for tokens in document_tokens for token in tokens})
document_count = len(document_tokens)
document_frequency = {
    term: sum(term in tokens for tokens in document_tokens)
    for term in vocabulary
}
idf = {
    term: math.log((1 + document_count) / (1 + document_frequency[term])) + 1
    for term in vocabulary
}

def tfidf_vector(tokens):
    counts = Counter(tokens)
    token_count = max(len(tokens), 1)
    return np.array(
        [(counts[term] / token_count) * idf[term] for term in vocabulary],
        dtype=float,
    )

query_vector = tfidf_vector(query_tokens)
document_matrix = np.vstack([tfidf_vector(tokens) for tokens in document_tokens])

print(f"Tamaño del vocabulario: {len(vocabulary)} términos")
print(f"Dimensión del vector de la query: {query_vector.shape}")
print(f"Dimensión de la matriz documental: {document_matrix.shape}")
print("Primeros términos del vocabulario:", vocabulary[:20])

Tamaño del vocabulario: 22 términos
Dimensión del vector de la query: (22,)
Dimensión de la matriz documental: (4, 22)
Primeros términos del vocabulario: ['alimentos', 'causan', 'demostración', 'deportivo', 'después', 'digestivos', 'documento', 'efectos', 'entrenamiento', 'estomacal', 'fútbol', 'generales', 'malestar', 'otros', 'partido', 'recibir', 'relación', 'resultados', 'secundarios', 'si']


## 4. Comparación mediante similitud coseno

La similitud coseno compara la dirección de los vectores de la query y de cada documento:

$$
sim(q,d) = \frac{q \cdot d}{||q|| \ ||d||}
$$

Un valor alto significa que los textos comparten una representación léxica similar en esta demostración. No equivale a veracidad.

In [5]:
def cosine_similarity(vector_a, vector_b):
    denominator = np.linalg.norm(vector_a) * np.linalg.norm(vector_b)
    if denominator == 0:
        return 0.0
    return float(np.dot(vector_a, vector_b) / denominator)

scores = [
    cosine_similarity(query_vector, document_vector)
    for document_vector in document_matrix
]

ranking = sorted(
    [
        {
            "position": None,
            "document_id": document["id"],
            "title": document["title"],
            "cosine_similarity": score,
        }
        for document, score in zip(documents, scores)
    ],
    key=lambda row: row["cosine_similarity"],
    reverse=True,
)

for position, row in enumerate(ranking, start=1):
    row["position"] = position

print(f"{'Pos.':<6}{'ID':<8}{'Similitud coseno':<20}Título")
print("-" * 90)
for row in ranking:
    print(
        f"{row['position']:<6}{row['document_id']:<8}"
        f"{row['cosine_similarity']:<20.4f}{row['title']}"
    )

Pos.  ID      Similitud coseno    Título
------------------------------------------------------------------------------------------
1     E1      0.6236              Evidencia demo sobre vacunas y molestias digestivas
2     E3      0.4950              Evidencia demo sobre alimentación y malestar
3     E2      0.0976              Evidencia demo sobre efectos secundarios de vacunación
4     E4      0.0000              Evidencia demo sobre deporte


In [6]:
# Tabla opcional: si pandas está disponible, se muestra una tabla navegable.
try:
    import pandas as pd
    from IPython.display import display
except ImportError:
    pd = None

if pd is not None:
    ranking_df = pd.DataFrame(ranking)
    display(ranking_df)
else:
    print("Pandas no está instalado; se conserva la tabla de texto anterior.")

,position,document_id,title,cosine_similarity
0,1,E1,Evidencia demo sobre vacunas y molestias diges...,0.623563
1,2,E3,Evidencia demo sobre alimentación y malestar,0.494984
2,3,E2,Evidencia demo sobre efectos secundarios de va...,0.097602
3,4,E4,Evidencia demo sobre deporte,0.000000


## Lectura del resultado

En esta simulación, TF-IDF premia la coincidencia literal de términos. Por eso un documento que menciona `malestar estomacal` puede obtener un puntaje alto aunque su contexto indique que no está relacionado con vacunas. Este comportamiento ilustra por qué Seminario 1 compara una línea base léxica con SBERT, que representa el significado de las oraciones mediante embeddings.

## 5. Validaciones de la minisimulación

Estas comprobaciones verifican que la query y los documentos están en el mismo espacio vectorial y que el ranking está ordenado por el puntaje calculado.

In [7]:
assert query_vector.ndim == 1
assert query_vector.shape[0] == len(vocabulary)
assert document_matrix.shape == (len(documents), len(vocabulary))
assert all(-1.0 - 1e-9 <= score <= 1.0 + 1e-9 for score in scores)
assert [row["document_id"] for row in ranking] == [
    row["document_id"]
    for row in sorted(ranking, key=lambda item: item["cosine_similarity"], reverse=True)
]
assert len(ranking) == len(documents)

print("Validaciones completadas correctamente.")
print("El resultado es un ranking de relación textual, no una predicción de veracidad.")

Validaciones completadas correctamente.
El resultado es un ranking de relación textual, no una predicción de veracidad.


## 6. Bloque opcional: SBERT real

La siguiente celda no es necesaria para ejecutar la simulación TF-IDF. Si se instala `sentence-transformers`, genera embeddings reales con un modelo multilingüe y vuelve a calcular el ranking mediante similitud coseno. Si el paquete o el modelo no están disponibles, la celda informa la situación sin interrumpir el notebook.

In [8]:
try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    print("SBERT opcional omitido: instala sentence-transformers para ejecutar esta sección.")
else:
    try:
        model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
        model = SentenceTransformer(model_name)
        sbert_query = model.encode([query], normalize_embeddings=True)[0]
        sbert_documents = model.encode(
            [document["text"] for document in documents],
            normalize_embeddings=True,
        )
        sbert_scores = np.dot(sbert_documents, sbert_query)
        sbert_ranking = sorted(
            [
                {
                    "position": None,
                    "document_id": document["id"],
                    "title": document["title"],
                    "cosine_similarity": float(score),
                }
                for document, score in zip(documents, sbert_scores)
            ],
            key=lambda row: row["cosine_similarity"],
            reverse=True,
        )
        for position, row in enumerate(sbert_ranking, start=1):
            row["position"] = position
        print(f"Ranking SBERT con {model_name}:")
        for row in sbert_ranking:
            print(
                f"{row['position']}. {row['document_id']} - "
                f"{row['cosine_similarity']:.4f} - {row['title']}"
            )
    except Exception as error:
        print("SBERT opcional no ejecutado; la simulación principal sigue disponible.")
        print(f"Detalle: {type(error).__name__}: {error}")

/Users/renzo/Documents/GitHub/recomendador-fuentes-confiables-nlp-xai/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6091.92it/s]


Ranking SBERT con sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2:
1. E1 - 0.8607 - Evidencia demo sobre vacunas y molestias digestivas
2. E3 - 0.6978 - Evidencia demo sobre alimentación y malestar
3. E2 - 0.6727 - Evidencia demo sobre efectos secundarios de vacunación
4. E4 - 0.0165 - Evidencia demo sobre deporte


## Interpretación y siguiente paso

El resultado muestra cómo una query se transforma en una representación numérica y cómo se ordenan documentos candidatos por cercanía. Para el experimento real de Seminario 1, el corpus sintético debe reemplazarse por evidencias validadas y el ranking debe evaluarse con Precision@k, Recall@k, MRR y nDCG@k.

La consulta a fuentes externas, la reindexación de nuevas evidencias y RAG pertenecen al desarrollo posterior de Seminario 2.

## 7. Evaluación del ranking

Estas métricas evalúan si las evidencias relevantes aparecen en las primeras posiciones del ranking. Los siguientes juicios de relevancia son sintéticos y solo sirven para demostrar el cálculo: E1 es altamente relevante, E2 es parcialmente relevante y E3/E4 no son relevantes para la query. En el experimento real deben reemplazarse por qrels revisados por personas.

La evaluación mide recuperación de evidencias; no determina si una afirmación es verdadera o falsa.

In [9]:
# Juicios de relevancia graduada para esta demostración.
relevance_judgments = {
    "E1": 2,  # Muy relevante: aborda vacunas y malestar estomacal.
    "E2": 1,  # Parcialmente relevante: aborda efectos secundarios de vacunas.
    "E3": 0,  # Comparte términos, pero no relaciona el malestar con vacunas.
    "E4": 0,  # Irrelevante para la query.
}

ranked_ids = [row["document_id"] for row in ranking]
graded_relevances = [relevance_judgments[document_id] for document_id in ranked_ids]

def precision_at_k(relevances, k):
    """Proporción de documentos relevantes dentro de las primeras k posiciones."""
    top_k = relevances[:k]
    return sum(relevance > 0 for relevance in top_k) / k if k else 0.0

def recall_at_k(relevances, k):
    """Proporción de todos los documentos relevantes recuperados en Top-k."""
    total_relevant = sum(relevance > 0 for relevance in relevances)
    return (
        sum(relevance > 0 for relevance in relevances[:k]) / total_relevant
        if total_relevant
        else 0.0
    )

def reciprocal_rank(relevances):
    """Recíproco de la posición del primer documento relevante."""
    for position, relevance in enumerate(relevances, start=1):
        if relevance > 0:
            return 1.0 / position
    return 0.0

def dcg_at_k(relevances, k):
    """DCG con ganancias graduadas: 2^relevancia - 1."""
    return sum(
        (2**relevance - 1) / np.log2(position + 2)
        for position, relevance in enumerate(relevances[:k])
    )

def ndcg_at_k(relevances, k):
    """DCG normalizado respecto del ranking ideal para la misma query."""
    ideal_relevances = sorted(relevance_judgments.values(), reverse=True)
    ideal_dcg = dcg_at_k(ideal_relevances, k)
    return dcg_at_k(relevances, k) / ideal_dcg if ideal_dcg else 0.0

mrr_value = reciprocal_rank(graded_relevances)
K_VALUES = [1, 2, 3, 4]
metric_rows = []
for k in K_VALUES:
    metric_rows.append({
        "k": k,
        "Precision@k": precision_at_k(graded_relevances, k),
        "Recall@k": recall_at_k(graded_relevances, k),
        "nDCG@k": ndcg_at_k(graded_relevances, k),
    })

print("Ranking evaluado:", ranked_ids)
print("Relevancias:   ", graded_relevances)
print()
print("Resultados de evaluación por k")
print("k  Precision@k  Recall@k  nDCG@k")
for row in metric_rows:
    print(
        f"{row['k']}  {row['Precision@k']:<11.4f} "
        f"{row['Recall@k']:<9.4f} {row['nDCG@k']:.4f}"
    )
print(f"\nMRR (independiente de k): {mrr_value:.4f}")

try:
    evaluation_df = pd.DataFrame(metric_rows)
    display(evaluation_df)
except NameError:
    pass

# Validaciones básicas de la evaluación.
assert set(ranked_ids) == set(relevance_judgments)
assert all(0.0 <= row["Precision@k"] <= 1.0 for row in metric_rows)
assert all(0.0 <= row["Recall@k"] <= 1.0 for row in metric_rows)
assert all(0.0 <= row["nDCG@k"] <= 1.0 for row in metric_rows)
assert 0.0 <= mrr_value <= 1.0
assert metric_rows[0]["Precision@k"] == 1.0
assert metric_rows[2]["Recall@k"] == 1.0
print("Métricas calculadas y validadas correctamente.")

Ranking evaluado: ['E1', 'E3', 'E2', 'E4']
Relevancias:    [2, 0, 1, 0]

Resultados de evaluación por k
k  Precision@k  Recall@k  nDCG@k
1  1.0000      0.5000    1.0000
2  0.5000      0.5000    0.8262
3  0.6667      1.0000    0.9639
4  0.5000      1.0000    0.9639

MRR (independiente de k): 1.0000


,k,Precision@k,Recall@k,nDCG@k
0,1,1.000000,0.5,1.000000
1,2,0.500000,0.5,0.826235
2,3,0.666667,1.0,0.963940
3,4,0.500000,1.0,0.963940


Métricas calculadas y validadas correctamente.


## 8. Simulación comparativa completa del workflow

Esta sección ejecuta los cuatro componentes conceptuales del workflow:

1. TF-IDF + similitud coseno.
2. BM25.
3. SBERT preentrenado, si se habilita la sección opcional.
4. SBERT ajustado con Triplet Loss, con una simulación didáctica reproducible en NumPy y una implementación real opcional.

La proyección lineal entrenada con NumPy no debe reportarse como SBERT. Su función es mostrar qué significa entrenar una representación con tripletas query–evidencia positiva–evidencia negativa. La evaluación compara rankings de evidencias, no clasifica la veracidad de las afirmaciones.

In [10]:
# Simulación comparativa completa del workflow de Seminario 1.
# Se reutilizan query, documents y normalize_and_tokenize definidos arriba.

SEED = 42
rng = np.random.default_rng(SEED)
documents_by_id = {document["id"]: document for document in documents}
document_ids = [document["id"] for document in documents]

DEMO_QRELS = {
    "E1": 2,  # Muy relevante.
    "E2": 1,  # Parcialmente relevante.
    "E3": 0,  # Coincidencia léxica, pero relación incorrecta.
    "E4": 0,  # Irrelevante.
}

def build_ranking(ids, scores, score_name):
    rows = [
        {
            "document_id": document_id,
            "title": documents_by_id[document_id]["title"],
            score_name: float(score),
        }
        for document_id, score in zip(ids, scores)
    ]
    rows.sort(key=lambda row: row[score_name], reverse=True)
    for position, row in enumerate(rows, start=1):
        row["position"] = position
    return rows

# ---------------------------------------------------------------------------
# 1. TF-IDF + similitud coseno
# ---------------------------------------------------------------------------
tfidf_ranking = [
    {
        "document_id": row["document_id"],
        "title": row["title"],
        "score": row["cosine_similarity"],
        "position": row["position"],
    }
    for row in ranking
]

# ---------------------------------------------------------------------------
# 2. BM25
# BM25 produce una puntuación propia; no se interpreta como coseno.
# ---------------------------------------------------------------------------
def bm25_scores(query_tokens, collection_tokens, k1=1.5, b=0.75):
    document_count = len(collection_tokens)
    average_length = np.mean([len(tokens) for tokens in collection_tokens])
    document_frequency = Counter(
        token
        for tokens in collection_tokens
        for token in set(tokens)
    )
    scores = []

    for tokens in collection_tokens:
        term_counts = Counter(tokens)
        document_length = len(tokens)
        score = 0.0

        for term in query_tokens:
            frequency = term_counts[term]
            if frequency == 0:
                continue

            df = document_frequency[term]
            inverse_document_frequency = math.log(
                1 + (document_count - df + 0.5) / (df + 0.5)
            )
            normalization = k1 * (
                1 - b + b * document_length / max(average_length, 1.0)
            )
            score += inverse_document_frequency * (
                frequency * (k1 + 1) / (frequency + normalization)
            )

        scores.append(score)

    return scores

bm25_ranking = build_ranking(
    document_ids,
    bm25_scores(query_tokens, document_tokens),
    "score",
)

print("Ranking BM25:")
for row in bm25_ranking:
    print(
        f"{row['position']}. {row['document_id']} - "
        f"{row['score']:.4f} - {row['title']}"
    )

# ---------------------------------------------------------------------------
# 3. Modelo entrenable didáctico con Triplet Loss
# Este bloque sí entrena una representación pequeña con NumPy.
# No es SBERT: es una simulación pedagógica del mecanismo de ajuste.
# ---------------------------------------------------------------------------
training_queries = [
    query,
    "¿Qué efectos secundarios pueden aparecer después de una vacuna?",
    "¿Qué alimentos causan malestar estomacal?",
]

training_triplets = [
    (training_queries[0], "E1", "E3"),  # hard negative léxico
    (training_queries[0], "E2", "E4"),
    (training_queries[1], "E2", "E4"),
    (training_queries[2], "E3", "E4"),
]

all_training_texts = training_queries + [
    document["text"] for document in documents
]
tiny_vocabulary = sorted(
    {
        token
        for text in all_training_texts
        for token in normalize_and_tokenize(text)
    }
)

def tiny_bow(text):
    tokens = normalize_and_tokenize(text)
    counts = Counter(tokens)
    vector = np.array(
        [counts[token] for token in tiny_vocabulary],
        dtype=float,
    )
    norm = np.linalg.norm(vector)
    return vector / norm if norm else vector

def projected_embedding(vector, weights):
    return vector @ weights

def normalized_cosine(vector_a, vector_b):
    norm_product = np.linalg.norm(vector_a) * np.linalg.norm(vector_b)
    return float(np.dot(vector_a, vector_b) / norm_product) if norm_product else 0.0

def triplet_loss_and_gradient(query_vector, positive_vector, negative_vector, weights, margin=0.5):
    query_embedding = projected_embedding(query_vector, weights)
    positive_embedding = projected_embedding(positive_vector, weights)
    negative_embedding = projected_embedding(negative_vector, weights)

    positive_distance = np.sum((query_embedding - positive_embedding) ** 2)
    negative_distance = np.sum((query_embedding - negative_embedding) ** 2)
    loss = max(0.0, positive_distance - negative_distance + margin)

    if loss == 0.0:
        return loss, np.zeros_like(weights)

    gradient_query = 2 * (negative_embedding - positive_embedding)
    gradient_positive = 2 * (positive_embedding - query_embedding)
    gradient_negative = 2 * (query_embedding - negative_embedding)

    gradient = (
        np.outer(query_vector, gradient_query)
        + np.outer(positive_vector, gradient_positive)
        + np.outer(negative_vector, gradient_negative)
    )
    return loss, gradient

def train_tiny_triplet_encoder(epochs=250, learning_rate=0.05, output_dim=8, initial_weights=None):
    if initial_weights is None:
        weights = rng.normal(
            loc=0.0,
            scale=0.1,
            size=(len(tiny_vocabulary), output_dim),
        )
    else:
        weights = initial_weights.copy()
    history = []

    for _ in range(epochs):
        epoch_loss = 0.0

        for triplet_query, positive_id, negative_id in training_triplets:
            query_vector = tiny_bow(triplet_query)
            positive_vector = tiny_bow(documents_by_id[positive_id]["text"])
            negative_vector = tiny_bow(documents_by_id[negative_id]["text"])

            loss, gradient = triplet_loss_and_gradient(
                query_vector,
                positive_vector,
                negative_vector,
                weights,
            )
            weights -= learning_rate * gradient
            epoch_loss += loss

        history.append(epoch_loss)

    return weights, history

def tiny_ranking(weights):
    query_embedding = projected_embedding(tiny_bow(query), weights)
    scores = [
        normalized_cosine(
            query_embedding,
            projected_embedding(tiny_bow(document["text"]), weights),
        )
        for document in documents
    ]
    return build_ranking(document_ids, scores, "score")

initial_weights = rng.normal(
    loc=0.0,
    scale=0.1,
    size=(len(tiny_vocabulary), 8),
)
trained_weights, loss_history = train_tiny_triplet_encoder(
    initial_weights=initial_weights,
)

tiny_before_ranking = tiny_ranking(initial_weights)
tiny_after_ranking = tiny_ranking(trained_weights)

print(
    "Pérdida Triplet Loss:",
    f"{loss_history[0]:.4f} → {loss_history[-1]:.4f}",
)
print("Simulación didáctica antes del ajuste:")
for row in tiny_before_ranking:
    print(f"{row['position']}. {row['document_id']} - {row['score']:.4f}")
print("Simulación didáctica después del ajuste:")
for row in tiny_after_ranking:
    print(f"{row['position']}. {row['document_id']} - {row['score']:.4f}")

# ---------------------------------------------------------------------------
# 4. Métricas de recuperación
# ---------------------------------------------------------------------------
def ranking_metrics(ranked_ids, qrels, k_values=(1, 2, 3, 4)):
    relevances = [qrels[document_id] for document_id in ranked_ids]
    total_relevant = sum(relevance > 0 for relevance in qrels.values())

    def precision_at_k(k):
        return sum(relevance > 0 for relevance in relevances[:k]) / k

    def recall_at_k(k):
        return (
            sum(relevance > 0 for relevance in relevances[:k]) / total_relevant
            if total_relevant
            else 0.0
        )

    def dcg_at_k(values, k):
        return sum(
            (2**relevance - 1) / np.log2(position + 2)
            for position, relevance in enumerate(values[:k])
        )

    ideal_relevances = sorted(qrels.values(), reverse=True)

    def ndcg_at_k(k):
        ideal_dcg = dcg_at_k(ideal_relevances, k)
        return dcg_at_k(relevances, k) / ideal_dcg if ideal_dcg else 0.0

    reciprocal_rank_value = next(
        (
            1.0 / position
            for position, relevance in enumerate(relevances, start=1)
            if relevance > 0
        ),
        0.0,
    )

    return {
        "MRR": reciprocal_rank_value,
        **{
            metric_name: values
            for metric_name, values in [
                (
                    "Precision@k",
                    {
                        k: precision_at_k(k)
                        for k in k_values
                    },
                ),
                (
                    "Recall@k",
                    {
                        k: recall_at_k(k)
                        for k in k_values
                    },
                ),
                (
                    "nDCG@k",
                    {
                        k: ndcg_at_k(k)
                        for k in k_values
                    },
                ),
            ]
        },
    }

method_rankings = {
    "TF-IDF + coseno": [row["document_id"] for row in tfidf_ranking],
    "BM25": [row["document_id"] for row in bm25_ranking],
    "Triplet Loss didáctico": [
        row["document_id"] for row in tiny_after_ranking
    ],
}

comparison_rows = []
for method_name, ranked_ids in method_rankings.items():
    metrics = ranking_metrics(ranked_ids, DEMO_QRELS)
    comparison_rows.append(
        {
            "Enfoque": method_name,
            "Ranking": " > ".join(ranked_ids),
            "P@1": metrics["Precision@k"][1],
            "R@1": metrics["Recall@k"][1],
            "P@3": metrics["Precision@k"][3],
            "R@3": metrics["Recall@k"][3],
            "nDCG@3": metrics["nDCG@k"][3],
            "MRR": metrics["MRR"],
        }
    )

print("\nComparación auxiliar sin SBERT real:")
for row in comparison_rows:
    print(
        f"{row['Enfoque']}: {row['Ranking']} | "
        f"P@3={row['P@3']:.4f}, "
        f"R@3={row['R@3']:.4f}, "
        f"nDCG@3={row['nDCG@3']:.4f}, "
        f"MRR={row['MRR']:.4f}"
    )

if "pd" in globals() and pd is not None:
    display(pd.DataFrame(comparison_rows))

assert len(method_rankings) == 3
assert all(set(ids) == set(document_ids) for ids in method_rankings.values())
assert all(0.0 <= row["P@3"] <= 1.0 for row in comparison_rows)
assert all(0.0 <= row["R@3"] <= 1.0 for row in comparison_rows)
assert all(0.0 <= row["nDCG@3"] <= 1.0 for row in comparison_rows)
assert all(0.0 <= row["MRR"] <= 1.0 for row in comparison_rows)
assert loss_history[-1] <= loss_history[0]
print("Validación de TF-IDF, BM25 y proxy didáctico de Triplet Loss completada.")

# ---------------------------------------------------------------------------
# 5. SBERT real opcional
# Para habilitarlo, instala sentence-transformers y cambia el indicador a True.
# ---------------------------------------------------------------------------
RUN_REAL_SBERT = True
real_sbert_rankings = {}
real_sbert_errors = {}

if not RUN_REAL_SBERT:
    print(
        "SBERT real omitido. Cambia RUN_REAL_SBERT a True e instala "
        "sentence-transformers para ejecutarlo."
    )
else:
    try:
        import copy
        from sentence_transformers import InputExample, SentenceTransformer, losses
        from torch.utils.data import DataLoader

        model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
        pretrained_model = SentenceTransformer(model_name)
        query_embedding = pretrained_model.encode(
            [query],
            normalize_embeddings=True,
        )[0]
        document_embeddings = pretrained_model.encode(
            [document["text"] for document in documents],
            normalize_embeddings=True,
        )
        pretrained_scores = np.dot(document_embeddings, query_embedding)
        pretrained_ranking = build_ranking(
            document_ids,
            pretrained_scores,
            "score",
        )
        real_sbert_rankings["SBERT preentrenado"] = [
            row["document_id"] for row in pretrained_ranking
        ]

        # Desde sentence-transformers 6.x, fit() necesita el paquete datasets.
        # Se valida aquí para conservar el resultado del SBERT preentrenado
        # y reportar por separado el fallo del ajuste.
        try:
            import datasets  # noqa: F401
        except ImportError as dependency_error:
            raise ImportError(
                "Falta 'datasets'. Instala el extra SBERT con "
                ".venv/bin/python -m pip install -e '.[sbert]'"
            ) from dependency_error

        train_examples = [
            InputExample(
                texts=[
                    triplet_query,
                    documents_by_id[positive_id]["text"],
                    documents_by_id[negative_id]["text"],
                ]
            )
            for triplet_query, positive_id, negative_id in training_triplets
        ]
        train_loader = DataLoader(train_examples, shuffle=True, batch_size=2)
        # Reutilizar el modelo ya cargado evita una segunda descarga o fallo
        # de red antes de iniciar el ajuste.
        fine_tuned_model = copy.deepcopy(pretrained_model)
        triplet_loss = losses.TripletLoss(
            model=fine_tuned_model,
            distance_metric=losses.TripletDistanceMetric.COSINE,
            triplet_margin=0.5,
        )
        fine_tuned_model.fit(
            train_objectives=[(train_loader, triplet_loss)],
            epochs=1,
            warmup_steps=0,
            show_progress_bar=False,
        )

        tuned_query_embedding = fine_tuned_model.encode(
            [query],
            normalize_embeddings=True,
        )[0]
        tuned_document_embeddings = fine_tuned_model.encode(
            [document["text"] for document in documents],
            normalize_embeddings=True,
        )
        tuned_scores = np.dot(
            tuned_document_embeddings,
            tuned_query_embedding,
        )
        tuned_ranking = build_ranking(document_ids, tuned_scores, "score")
        real_sbert_rankings["SBERT + Triplet Loss"] = [
            row["document_id"] for row in tuned_ranking
        ]

        print("SBERT real preentrenado:", real_sbert_rankings["SBERT preentrenado"])
        print("SBERT real ajustado:", real_sbert_rankings["SBERT + Triplet Loss"])
    except ImportError as error:
        detail = f"{type(error).__name__}: {error}"
        real_sbert_errors["SBERT + Triplet Loss"] = detail
        print(
            "SBERT real no disponible: verifica sentence-transformers y datasets "
            "para habilitar esta sección."
        )
        print(f"Detalle: {detail}")
    except Exception as error:
        real_sbert_errors["SBERT + Triplet Loss"] = f"{type(error).__name__}: {error}"
        print(
            "SBERT real no se pudo ejecutar; la simulación NumPy continúa."
        )
        print(f"Detalle: {real_sbert_errors['SBERT + Triplet Loss']}")

print(
    "Conclusión: TF-IDF y BM25 son baselines de recuperación; "
    "el bloque NumPy muestra el mecanismo de entrenamiento con Triplet Loss; "
    "la implementación real de SBERT es opcional."
)

Ranking BM25:
1. E1 - 2.6902 - Evidencia demo sobre vacunas y molestias digestivas
2. E3 - 1.8705 - Evidencia demo sobre alimentación y malestar
3. E2 - 0.3427 - Evidencia demo sobre efectos secundarios de vacunación
4. E4 - 0.0000 - Evidencia demo sobre deporte
Pérdida Triplet Loss: 1.8132 → 0.0000
Simulación didáctica antes del ajuste:
1. E1 - 0.7629
2. E3 - 0.3283
3. E2 - 0.2390
4. E4 - -0.1422
Simulación didáctica después del ajuste:
1. E1 - 0.9464
2. E2 - -0.0128
3. E3 - -0.6828
4. E4 - -0.8888

Comparación auxiliar sin SBERT real:
TF-IDF + coseno: E1 > E3 > E2 > E4 | P@3=0.6667, R@3=1.0000, nDCG@3=0.9639, MRR=1.0000
BM25: E1 > E3 > E2 > E4 | P@3=0.6667, R@3=1.0000, nDCG@3=0.9639, MRR=1.0000
Triplet Loss didáctico: E1 > E2 > E3 > E4 | P@3=0.6667, R@3=1.0000, nDCG@3=1.0000, MRR=1.0000


,Enfoque,Ranking,P@1,R@1,P@3,R@3,nDCG@3,MRR
0,TF-IDF + coseno,E1 > E3 > E2 > E4,1.0,0.5,0.666667,1.0,0.96394,1.0
1,BM25,E1 > E3 > E2 > E4,1.0,0.5,0.666667,1.0,0.96394,1.0
2,Triplet Loss didáctico,E1 > E2 > E3 > E4,1.0,0.5,0.666667,1.0,1.00000,1.0


Validación de TF-IDF, BM25 y proxy didáctico de Triplet Loss completada.


/var/folders/f_/f_wm3g5j59gbfp5kbvdx136r0000gn/T/ipykernel_94797/1794248464.py:360: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import InputExample, SentenceTransformer, losses
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 19216.01it/s]
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 0, 'pad_token_id': 1}.
/Users/renzo/Documents/GitHub/recomendador-fuentes-confiables-nlp-xai/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loa

{'train_runtime': '3.572', 'train_samples_per_second': '1.12', 'train_steps_per_second': '0.56', 'train_loss': '0.07267', 'epoch': '1'}
SBERT real preentrenado: ['E1', 'E3', 'E2', 'E4']
SBERT real ajustado: ['E1', 'E2', 'E3', 'E4']
Conclusión: TF-IDF y BM25 son baselines de recuperación; el bloque NumPy muestra el mecanismo de entrenamiento con Triplet Loss; la implementación real de SBERT es opcional.


## 9. Tabla final de los cuatro enfoques de la tesis

La comparación metodológica correcta contiene exactamente cuatro enfoques:

1. TF-IDF + similitud coseno.
2. BM25.
3. SBERT preentrenado.
4. SBERT ajustado con Triplet Loss.

La proyección lineal entrenada con NumPy es una demostración didáctica del mecanismo de Triplet Loss, pero no reemplaza a SBERT. Si SBERT real no está instalado o RUN_REAL_SBERT es False, sus filas aparecen como pendientes y no se inventan métricas.

In [11]:
# Tabla final con los cuatro enfoques definidos para Seminario 1.
FOUR_APPROACHES = [
    ("TF-IDF + coseno", "TF-IDF + coseno"),
    ("BM25", "BM25"),
    ("SBERT preentrenado", "SBERT preentrenado"),
    ("SBERT + Triplet Loss", "SBERT + Triplet Loss"),
]

four_approach_rows = []

for display_name, ranking_key in FOUR_APPROACHES:
    if ranking_key in {"TF-IDF + coseno", "BM25"}:
        ranked_ids = method_rankings[ranking_key]
    else:
        ranked_ids = real_sbert_rankings.get(ranking_key)

    if ranked_ids is None:
        error_detail = real_sbert_errors.get(ranking_key)
        status = (
            f"Error: {error_detail}"
            if error_detail
            else "Pendiente: activar SBERT real"
        )
        four_approach_rows.append(
            {
                "Enfoque": display_name,
                "Estado": status,
                "Ranking": "—",
                "P@1": None,
                "R@1": None,
                "P@3": None,
                "R@3": None,
                "nDCG@3": None,
                "MRR": None,
            }
        )
        continue

    metrics = ranking_metrics(ranked_ids, DEMO_QRELS)
    four_approach_rows.append(
        {
            "Enfoque": display_name,
            "Estado": "Ejecutado",
            "Ranking": " > ".join(ranked_ids),
            "P@1": metrics["Precision@k"][1],
            "R@1": metrics["Recall@k"][1],
            "P@3": metrics["Precision@k"][3],
            "R@3": metrics["Recall@k"][3],
            "nDCG@3": metrics["nDCG@k"][3],
            "MRR": metrics["MRR"],
        }
    )

print("Tabla metodológica final:")
if "pd" in globals() and pd is not None:
    display(pd.DataFrame(four_approach_rows))
else:
    for row in four_approach_rows:
        print(row)

assert len(four_approach_rows) == 4
assert [row["Enfoque"] for row in four_approach_rows] == [
    "TF-IDF + coseno",
    "BM25",
    "SBERT preentrenado",
    "SBERT + Triplet Loss",
]
assert sum(row["Estado"] == "Ejecutado" for row in four_approach_rows) >= 2

if any(row["Estado"].startswith("Pendiente") for row in four_approach_rows):
    print(
        "Hay cuatro enfoques definidos, pero SBERT real está pendiente. "
        "Instala sentence-transformers y cambia RUN_REAL_SBERT a True."
    )

Tabla metodológica final:


,Enfoque,Estado,Ranking,P@1,R@1,P@3,R@3,nDCG@3,MRR
0,TF-IDF + coseno,Ejecutado,E1 > E3 > E2 > E4,1.0,0.5,0.666667,1.0,0.96394,1.0
1,BM25,Ejecutado,E1 > E3 > E2 > E4,1.0,0.5,0.666667,1.0,0.96394,1.0
2,SBERT preentrenado,Ejecutado,E1 > E3 > E2 > E4,1.0,0.5,0.666667,1.0,0.96394,1.0
3,SBERT + Triplet Loss,Ejecutado,E1 > E2 > E3 > E4,1.0,0.5,0.666667,1.0,1.00000,1.0
